# Coding CC1 manual baseline

In [1]:
import sys
from pathlib import Path

repo_root = Path.cwd()
if not (repo_root / "data").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import pandas as pd

from evals_local.metrics import label_metrics
from evals_local.normalizers import to_label_set, normalize_threat_labels, normalize_region_labels, normalize_geo_labels

In [2]:
annotated_path = repo_root / "data/consistency-check-datasets/data-coding/annotated_coding_dataset.xlsx"
labels_root = repo_root / "data/labels"

sheet_to_truth = {
    "l1_manual_sample": labels_root / "l1" / "L1) driver set.csv",
    "l2_manual_sample": labels_root / "l2" / "L2) threats set.csv",
    "l3_manual_sample": labels_root / "l3" / "L3) geography set.csv",
    "l4_manual_sample": labels_root / "l4" / "L4) ecosystem set.csv",
    "l5_manual_sample": labels_root / "l5" / "L5) study set.csv",
    "l6_manual_sample": labels_root / "l6" / "L6) taxa set.csv",
}

In [3]:
def merge_manual_truth(sheet_name):
    manual = pd.read_excel(annotated_path, sheet_name=sheet_name).copy()
    truth = pd.read_csv(sheet_to_truth[sheet_name]).copy()
    manual["UT (Unique WOS ID)"] = manual["UT (Unique WOS ID)"].astype(str).str.strip()
    truth["UT (Unique WOS ID)"] = truth["UT (Unique WOS ID)"].astype(str).str.strip()
    return manual.merge(
        truth,
        on="UT (Unique WOS ID)",
        how="left",
        validate="one_to_one",
        suffixes=("_pred", ""),
    )


def weighted_row(df, truth_col, pred_col, normalizer, include_mask=None):
    metrics = label_metrics(df, truth_col, pred_col, normalizer=normalizer, include_mask=include_mask)
    return metrics[metrics["label"] == "_weighted"].iloc[0]


def no_banned_mask(df, cols, banned, lowercase=False):
    if lowercase:
        parts = [df[c].apply(lambda v: len({s.lower() for s in to_label_set(v)} & banned) == 0) for c in cols]
    else:
        parts = [df[c].apply(lambda v: len(to_label_set(v) & banned) == 0) for c in cols]
    return pd.concat(parts, axis=1).all(axis=1)

In [4]:
results = []
full_metrics = {}

# L1
df = merge_manual_truth("l1_manual_sample")
row = weighted_row(df, "driver", "driver_pred", to_label_set)
results.append(["L1", "driver", row["precision"], row["recall"], row["f1"], row["kappa"], row["support"], row["n"]])

# L2 (manual workbook currently covers threats_l0 only)
df = merge_manual_truth("l2_manual_sample")
mask = no_banned_mask(df, ["threats_l0", "threats_l0_pred"], {"no threat_l2 candidates found", "no threat_l1 candidates found", "unclear"})
row = weighted_row(df, "threats_l0", "threats_l0_pred", normalize_threat_labels, include_mask=mask)
results.append(["L2", "threats_l0", row["precision"], row["recall"], row["f1"], row["kappa"], row["support"], row["n"]])

# L3
df = merge_manual_truth("l3_manual_sample")
for truth_col, pred_col, normalizer in [
    ("region", "region_pred", normalize_region_labels),
    ("sub-region", "sub-region_pred", to_label_set),
    ("country", "country_pred", normalize_geo_labels),
]:
    row = weighted_row(df, truth_col, pred_col, normalizer)
    results.append(["L3", truth_col, row["precision"], row["recall"], row["f1"], row["kappa"], row["support"], row["n"]])

# L4
df = merge_manual_truth("l4_manual_sample")
row = weighted_row(df, "realm", "realm_pred", to_label_set)
results.append(["L4", "realm", row["precision"], row["recall"], row["f1"], row["kappa"], row["support"], row["n"]])
mask = no_banned_mask(df, ["biome", "biome_pred"], {"no biome candidates found"}, lowercase=True)
row = weighted_row(df, "biome", "biome_pred", to_label_set, include_mask=mask)
results.append(["L4", "biome", row["precision"], row["recall"], row["f1"], row["kappa"], row["support"], row["n"]])

# L5
df = merge_manual_truth("l5_manual_sample")
row = weighted_row(df, "study_design", "study_design_pred", to_label_set)
results.append(["L5", "study_design", row["precision"], row["recall"], row["f1"], row["kappa"], row["support"], row["n"]])

# L6
df = merge_manual_truth("l6_manual_sample")
for truth_col, pred_col in [
    ("kingdom", "kingdom_pred"),
    ("phylum", "phylum_pred"),
    ("class", "class_pred"),
    ("order", "order_pred"),
    ("genus", "genus_pred"),
    ("specie", "specie_pred"),
]:
    mask = no_banned_mask(df, [truth_col, pred_col], {"not applicable", "unclear"})
    row = weighted_row(df, truth_col, pred_col, to_label_set, include_mask=mask)
    results.append(["L6", truth_col, row["precision"], row["recall"], row["f1"], row["kappa"], row["support"], row["n"]])

summary = pd.DataFrame(results, columns=["task", "label_set", "precision", "recall", "f1", "kappa", "support", "n"])
summary

,task,label_set,precision,recall,f1,kappa,support,n
0,L1,driver,0.865955,0.794393,0.819442,0.760524,107,100
1,L2,threats_l0,0.822205,0.701493,0.750562,0.676622,134,86
2,L3,region,0.950163,0.901961,0.923781,0.895433,102,100
3,L3,sub-region,0.963530,0.913978,0.933274,0.908148,93,91
4,L3,country,0.950988,0.819149,0.873886,0.846118,94,90
5,L4,realm,1.000000,0.930693,0.961446,0.936470,101,100
6,L4,biome,1.000000,1.000000,1.000000,NaN,1,1
7,L5,study_design,0.823071,0.764706,0.767632,0.526580,102,100
8,L6,kingdom,0.991250,1.000000,0.995333,0.994957,100,100
9,L6,phylum,1.000000,1.000000,1.000000,0.000000,90,90


In [5]:
summary.assign(
    precision=(summary["precision"] * 100).round(2),
    recall=(summary["recall"] * 100).round(2),
    f1=(summary["f1"] * 100).round(2),
    kappa=(summary["kappa"] * 100).round(2),
)

,task,label_set,precision,recall,f1,kappa,support,n
0,L1,driver,86.60,79.44,81.94,76.05,107,100
1,L2,threats_l0,82.22,70.15,75.06,67.66,134,86
2,L3,region,95.02,90.20,92.38,89.54,102,100
3,L3,sub-region,96.35,91.40,93.33,90.81,93,91
4,L3,country,95.10,81.91,87.39,84.61,94,90
5,L4,realm,100.00,93.07,96.14,93.65,101,100
6,L4,biome,100.00,100.00,100.00,NaN,1,1
7,L5,study_design,82.31,76.47,76.76,52.66,102,100
8,L6,kingdom,99.12,100.00,99.53,99.50,100,100
9,L6,phylum,100.00,100.00,100.00,0.00,90,90
